In [1]:
import pandas as pd
import mysql.connector
from scipy.stats import ttest_ind

conn = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password="Vishnu2005v",
    database="olist_ecommerce"
)

print("MySQL connection successful!")

MySQL connection successful!


## Prepare T-Test Data

In [2]:
query = """
SELECT
    CASE
        WHEN o.order_delivered_customer_date <= o.order_estimated_delivery_date
            THEN 'On Time'
        ELSE 'Delayed'
    END AS delivery_status,
    r.review_score
FROM orders o
JOIN order_reviews r
    ON o.order_id = r.order_id
WHERE o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
"""

delivery_reviews = pd.read_sql(query, conn)

print(delivery_reviews.head())
print(delivery_reviews["delivery_status"].value_counts())

C:\Users\Vishnupriyan\AppData\Local\Temp\ipykernel_11204\3336914756.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  delivery_reviews = pd.read_sql(query, conn)


  delivery_status  review_score
0         On Time             5
1         On Time             4
2         On Time             5
3         On Time             4
4         On Time             5
delivery_status
On Time    88658
Delayed     7701
Name: count, dtype: int64


Compare the Groups

In [3]:
group_summary = (
    delivery_reviews
    .groupby("delivery_status")["review_score"]
    .agg(["count", "mean", "std"])
    .round(2)
)

print(group_summary)

                 count  mean   std
delivery_status                   
Delayed           7701  2.57  1.66
On Time          88658  4.29  1.15


## Independent T-Test

In [4]:
delayed_scores = delivery_reviews[
    delivery_reviews["delivery_status"] == "Delayed"
]["review_score"]

on_time_scores = delivery_reviews[
    delivery_reviews["delivery_status"] == "On Time"
]["review_score"]

t_stat, p_value = ttest_ind(
    delayed_scores,
    on_time_scores,
    equal_var=False
)

print("T-statistic:", round(t_stat, 4))
print("P-value:", p_value)

T-statistic: -89.5508
P-value: 0.0


### T-Test Conclusion

- Significance level (α): 0.05
- T-statistic: -89.5508
- P-value: < 0.001
- Decision: Reject the null hypothesis.

**Business Insight:** Delayed orders have a significantly lower average review score (2.57) than on-time orders (4.29). This indicates that delivery delays are strongly associated with lower customer satisfaction.

## ANOVA — Product Category & Spending

## Load Category and Order Value Data

In [5]:
query = """
SELECT
    p.product_category_name,
    oi.price
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
WHERE p.product_category_name IS NOT NULL
"""

category_spending = pd.read_sql(query, conn)

print(category_spending.head())
print(category_spending.shape)

C:\Users\Vishnupriyan\AppData\Local\Temp\ipykernel_11204\2353214782.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  category_spending = pd.read_sql(query, conn)


   product_category_name   price
0             perfumaria  101.65
1             automotivo  129.90
2        cama_mesa_banho  229.00
3  utilidades_domesticas   58.90
4  utilidades_domesticas   58.90
(112650, 2)


## Check Variance Assumption

In [6]:
from scipy.stats import levene

groups = [
    group["price"].values
    for _, group in category_spending.groupby("product_category_name")
]

levene_stat, levene_p = levene(*groups)

print("Levene statistic:", round(levene_stat, 4))
print("Levene p-value:", levene_p)

Levene statistic: 86.4833
Levene p-value: 0.0


## Run One-Way ANOVA

In [7]:
from scipy.stats import f_oneway

anova_groups = [
    group["price"].values
    for _, group in category_spending.groupby("product_category_name")
]

f_stat, p_value = f_oneway(*anova_groups)

print("F-statistic:", round(f_stat, 4))
print("P-value:", p_value)

F-statistic: 188.6077
P-value: 0.0


### ANOVA Conclusion

- Significance level (α): 0.05
- F-statistic: 188.6077
- P-value: < 0.001
- Decision: Reject the null hypothesis.

**Business Insight:** The average product price differs significantly across product categories. This indicates that customer spending varies significantly depending on the product category.

**Assumption Note:** Levene's test indicated unequal variances across categories (p < 0.05). Therefore, the standard ANOVA result should be interpreted with caution.

## Chi-Square — Payment Method & Order Status

In [8]:
query = """
SELECT
    op.payment_type,
    o.order_status
FROM order_payments op
JOIN orders o
    ON op.order_id = o.order_id
"""

payment_status = pd.read_sql(query, conn)

print(payment_status.head())
print(payment_status.shape)

C:\Users\Vishnupriyan\AppData\Local\Temp\ipykernel_11204\1464765768.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  payment_status = pd.read_sql(query, conn)


  payment_type order_status
0  credit_card    delivered
1  credit_card    delivered
2  credit_card    delivered
3  credit_card    delivered
4  credit_card    delivered
(103884, 2)


## Contingency Table

In [9]:
contingency_table = pd.crosstab(
    payment_status["payment_type"],
    payment_status["order_status"]
)

print(contingency_table)

order_status  approved  canceled  created  delivered  invoiced  processing  \
payment_type                                                                 
boleto               0        95        2      19191        67          70   
credit_card          2       444        3      74584       239         224   
debit_card           0         7        0       1486         6           2   
not_defined          0         3        0          0         0           0   
voucher              0       115        0       5493        13          23   

order_status  shipped  unavailable  
payment_type                        
boleto            209          150  
credit_card       851          446  
debit_card         22            6  
not_defined         0            0  
voucher            84           47  


## Chi-Square Test

In [10]:
from scipy.stats import chi2_contingency

chi2_stat, p_value, dof, expected = chi2_contingency(contingency_table)

print("Chi-Square statistic:", round(chi2_stat, 4))
print("Degrees of freedom:", dof)
print("P-value:", p_value)

Chi-Square statistic: 677.0648
Degrees of freedom: 28
P-value: 1.2154169508522127e-124


### Chi-Square Test Conclusion

- Significance level (α): 0.05
- Chi-Square statistic: 677.0648
- Degrees of freedom: 28
- P-value: < 0.001
- Decision: Reject the null hypothesis.

**Business Insight:** There is a statistically significant association between payment method and order status. This indicates that order outcomes are not independent of the payment method used by customers.